In [0]:
print("=" * 80)
print("GENERACIÓN DE SCORING DE RIESGO DE CRÉDITO POR CLIENTE")
print("=" * 80)

# Cargar dataset completo desde Gold Layer
df_scoring = spark.table("gold.credit_risk_features").toPandas()
print(f"\nTotal de clientes: {len(df_scoring):,}")

# Separar features y target
X_all = df_scoring.drop('target', axis=1)
y_all = df_scoring['target']

# Aplicar preprocesamiento
X_all_processed = preprocessor.transform(X_all)

# Generar predicciones con el mejor modelo
risk_probabilities = best_model.predict_proba(X_all_processed)[:, 1]
risk_predictions = best_model.predict(X_all_processed)

# Crear DataFrame de scoring
df_risk_scoring = df_scoring.copy()
df_risk_scoring['credit_risk_score'] = risk_probabilities
df_risk_scoring['predicted_default'] = risk_predictions
df_risk_scoring['actual_default'] = y_all

# Categorizar riesgo en niveles
def categorize_risk(score):
    if score < 0.2:
        return 'Muy Bajo'
    elif score < 0.4:
        return 'Bajo'
    elif score < 0.6:
        return 'Medio'
    elif score < 0.8:
        return 'Alto'
    else:
        return 'Muy Alto'

df_risk_scoring['risk_category'] = df_risk_scoring['credit_risk_score'].apply(categorize_risk)

# Convertir score a escala 0-1000 (como credit score tradicional)
df_risk_scoring['credit_score_1000'] = (1 - df_risk_scoring['credit_risk_score']) * 1000
df_risk_scoring['credit_score_1000'] = df_risk_scoring['credit_score_1000'].round(0).astype(int)

print("\n✓ Scoring generado exitosamente\n")

# Estadísticas del scoring
print("=== ESTADÍSTICAS DEL SCORING ===")
print(f"\nScore de riesgo (probabilidad de default):")
print(f"  - Mínimo: {df_risk_scoring['credit_risk_score'].min():.4f}")
print(f"  - Máximo: {df_risk_scoring['credit_risk_score'].max():.4f}")
print(f"  - Promedio: {df_risk_scoring['credit_risk_score'].mean():.4f}")
print(f"  - Mediana: {df_risk_scoring['credit_risk_score'].median():.4f}")

print(f"\nDistribución por categoría de riesgo:")
risk_dist = df_risk_scoring['risk_category'].value_counts().sort_index()
for cat, count in risk_dist.items():
    pct = count / len(df_risk_scoring) * 100
    print(f"  {cat:12s}: {count:5d} clientes ({pct:5.2f}%)")

# Visualizaciones
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Distribución del score de riesgo
axes[0, 0].hist(df_risk_scoring['credit_risk_score'], bins=50, color='steelblue', alpha=0.7, edgecolor='black')
axes[0, 0].axvline(df_risk_scoring['credit_risk_score'].mean(), color='red', linestyle='--', linewidth=2, label=f'Media: {df_risk_scoring["credit_risk_score"].mean():.3f}')
axes[0, 0].set_xlabel('Score de Riesgo (Probabilidad de Default)', fontweight='bold')
axes[0, 0].set_ylabel('Frecuencia', fontweight='bold')
axes[0, 0].set_title('Distribución del Score de Riesgo de Crédito', fontsize=12, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# 2. Distribución por categoría de riesgo
risk_order = ['Muy Bajo', 'Bajo', 'Medio', 'Alto', 'Muy Alto']
risk_counts = [risk_dist.get(cat, 0) for cat in risk_order]
colors_risk = ['green', 'yellowgreen', 'yellow', 'orange', 'red']
axes[0, 1].bar(risk_order, risk_counts, color=colors_risk, alpha=0.7, edgecolor='black')
axes[0, 1].set_xlabel('Categoría de Riesgo', fontweight='bold')
axes[0, 1].set_ylabel('Número de Clientes', fontweight='bold')
axes[0, 1].set_title('Clientes por Categoría de Riesgo', fontsize=12, fontweight='bold')
axes[0, 1].tick_params(axis='x', rotation=45)
for i, v in enumerate(risk_counts):
    axes[0, 1].text(i, v + 50, str(v), ha='center', fontweight='bold')
axes[0, 1].grid(axis='y', alpha=0.3)

# 3. Credit Score (escala 0-1000)
axes[1, 0].hist(df_risk_scoring['credit_score_1000'], bins=50, color='purple', alpha=0.7, edgecolor='black')
axes[1, 0].axvline(df_risk_scoring['credit_score_1000'].mean(), color='red', linestyle='--', linewidth=2, label=f'Media: {df_risk_scoring["credit_score_1000"].mean():.0f}')
axes[1, 0].set_xlabel('Credit Score (0-1000)', fontweight='bold')
axes[1, 0].set_ylabel('Frecuencia', fontweight='bold')
axes[1, 0].set_title('Distribución del Credit Score (Escala 0-1000)', fontsize=12, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# 4. Comparación: Predicho vs Real
comparison = pd.crosstab(df_risk_scoring['actual_default'], df_risk_scoring['predicted_default'], normalize='index') * 100
sns.heatmap(comparison, annot=True, fmt='.1f', cmap='RdYlGn_r', ax=axes[1, 1], cbar_kws={'label': 'Porcentaje'})
axes[1, 1].set_title('Predicción vs Realidad (% por fila)', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Predicción')
axes[1, 1].set_ylabel('Realidad')
axes[1, 1].set_xticklabels(['No Default', 'Default'])
axes[1, 1].set_yticklabels(['No Default', 'Default'])

plt.tight_layout()
plt.show()

# Ejemplos de clientes por categoría
print("\n=== EJEMPLOS DE CLIENTES POR CATEGORÍA DE RIESGO ===")
for cat in ['Muy Bajo', 'Bajo', 'Medio', 'Alto', 'Muy Alto']:
    sample = df_risk_scoring[df_risk_scoring['risk_category'] == cat].head(2)
    if len(sample) > 0:
        print(f"\n{cat.upper()}:")
        for idx, row in sample.iterrows():
            print(f"  - Score: {row['credit_risk_score']:.4f} | Credit Score (0-1000): {row['credit_score_1000']} | Default Real: {'Sí' if row['actual_default'] == 1 else 'No'}")

print("\n" + "=" * 80)

In [0]:
# Seleccionar columnas relevantes para la tabla final
scoring_columns = ['credit_risk_score', 'credit_score_1000', 'risk_category', 'predicted_default', 'actual_default']

# Añadir algunas features importantes para contexto
important_features = []
if 'person_age' in df_risk_scoring.columns:
    important_features.append('person_age')
if 'person_income' in df_risk_scoring.columns:
    important_features.append('person_income')
if 'loan_amnt' in df_risk_scoring.columns:
    important_features.append('loan_amnt')
if 'loan_int_rate' in df_risk_scoring.columns:
    important_features.append('loan_int_rate')
if 'person_emp_length' in df_risk_scoring.columns:
    important_features.append('person_emp_length')

final_columns = important_features + scoring_columns
df_final_scoring = df_risk_scoring[final_columns].copy()

# Añadir timestamp del scoring
df_final_scoring['scoring_timestamp'] = datetime.now()

# Crear ID de cliente (si no existe)
df_final_scoring.insert(0, 'customer_id', range(1, len(df_final_scoring) + 1))

# Guardar en Delta como tabla Gold
df_final_scoring_spark = spark.createDataFrame(df_final_scoring)
df_final_scoring_spark.write.format("delta").mode("overwrite").saveAsTable("gold.credit_risk_scoring")

print("=" * 80)
print("✓ SCORING FINAL GUARDADO EN GOLD LAYER")
print("=" * 80)
print(f"\nTabla: gold.credit_risk_scoring")
print(f"Registros: {len(df_final_scoring):,}")
print(f"Columnas: {len(df_final_scoring.columns)}")
print(f"\nColumnas guardadas:")
for i, col in enumerate(df_final_scoring.columns, 1):
    print(f"  {i:2d}. {col}")

print("\n=== MUESTRA DE LA TABLA FINAL ===")
display(df_final_scoring.head(10))

print("\n=== RESUMEN FINAL ===")
print(f"✓ Arquitectura Medallion implementada:")
print(f"  - Bronze: bronze.credit_risk_raw (datos crudos)")
print(f"  - Silver: silver.credit_risk_cleaned (datos limpios + features)")
print(f"  - Gold: gold.credit_risk_features (features para ML)")
print(f"  - Gold: gold.credit_risk_scoring (scoring final por cliente)")
print(f"\n✓ Modelos comparados: {len(results_df)}")
print(f"\n⭐ Mejor modelo: {best_model_name}")
print(f"   - AUC en Validación: {best_model_info['val_auc']:.4f}")
print(f"   - AUC en Test: {test_auc:.4f}")
print(f"\n✓ Scoring generado para {len(df_final_scoring):,} clientes")
print(f"\n=" * 80)
print("PROYECTO COMPLETADO CON ÉXITO")
print("=" * 80)

In [0]:
# Crear DataFrame con resultados
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('val_auc', ascending=False)

print("=" * 80)
print("COMPARACIÓN DE MODELOS - AUC (Area Under the Curve)")
print("=" * 80)
print("\nRanking por Validation AUC (de mayor a menor):\n")
print(results_df[['modelo', 'train_auc', 'val_auc']].to_string(index=False))
print("\n" + "=" * 80)

# Identificar el mejor modelo
best_model_info = results_df.iloc[0]
print(f"\n⭐ MEJOR MODELO: {best_model_info['modelo']}")
print(f"   - Validation AUC: {best_model_info['val_auc']:.4f}")
print(f"   - Train AUC: {best_model_info['train_auc']:.4f}")
print(f"   - Overfitting: {best_model_info['train_auc'] - best_model_info['val_auc']:.4f}")
print(f"   - Run ID: {best_model_info['run_id']}")

# Visualización comparativa
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico 1: AUC por modelo
x_pos = np.arange(len(results_df))
width = 0.35

ax1.bar(x_pos - width/2, results_df['train_auc'], width, label='Train AUC', alpha=0.8, color='skyblue')
ax1.bar(x_pos + width/2, results_df['val_auc'], width, label='Validation AUC', alpha=0.8, color='orange')
ax1.set_xlabel('Modelo', fontweight='bold')
ax1.set_ylabel('AUC Score', fontweight='bold')
ax1.set_title('Comparación de Modelos - AUC', fontsize=14, fontweight='bold')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(results_df['modelo'], rotation=45, ha='right')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim([0.5, 1.0])

# Agregar valores sobre las barras
for i, (train, val) in enumerate(zip(results_df['train_auc'], results_df['val_auc'])):
    ax1.text(i - width/2, train + 0.01, f'{train:.3f}', ha='center', va='bottom', fontsize=9)
    ax1.text(i + width/2, val + 0.01, f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Gráfico 2: Diferencia Train-Val (Overfitting)
results_df['overfitting'] = results_df['train_auc'] - results_df['val_auc']
colors = ['red' if x > 0.05 else 'green' for x in results_df['overfitting']]
ax2.barh(results_df['modelo'], results_df['overfitting'], color=colors, alpha=0.7)
ax2.set_xlabel('Train AUC - Val AUC', fontweight='bold')
ax2.set_title('Análisis de Overfitting', fontsize=14, fontweight='bold')
ax2.axvline(x=0.05, color='red', linestyle='--', linewidth=2, label='Umbral overfitting (0.05)')
ax2.legend()
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Comparación completada")

In [0]:
# Cargar el mejor modelo desde MLflow
best_model_name = best_model_info['modelo']
print(f"=== EVALUACIÓN FINAL DEL MEJOR MODELO: {best_model_name} ===")
print(f"Run ID: {best_model_info['run_id']}\n")

# Seleccionar el modelo entrenado según el nombre
if best_model_name == 'Logistic Regression':
    best_model = model_lr
elif best_model_name == 'Random Forest':
    best_model = model_rf
elif best_model_name == 'Gradient Boosting':
    best_model = model_gb
elif best_model_name == 'XGBoost':
    best_model = model_xgb
elif best_model_name == 'LightGBM':
    best_model = model_lgb

# Predicciones en Test Set
y_test_pred_proba = best_model.predict_proba(X_test_processed)[:, 1]
y_test_pred = best_model.predict(X_test_processed)

# Métricas en Test Set
test_auc = roc_auc_score(y_test, y_test_pred_proba)

print("Métricas en Test Set:")
print(f"  AUC Score: {test_auc:.4f}")
print(f"\nReporte de Clasificación:")
print(classification_report(y_test, y_test_pred, target_names=['No Default', 'Default']))

# Matriz de confusión
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de confusión
cm = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1, cbar=False)
ax1.set_title(f'Matriz de Confusión - {best_model_name}', fontsize=12, fontweight='bold')
ax1.set_ylabel('Valor Real')
ax1.set_xlabel('Predicción')
ax1.set_xticklabels(['No Default', 'Default'])
ax1.set_yticklabels(['No Default', 'Default'])

# Curva ROC
fpr, tpr, thresholds = roc_curve(y_test, y_test_pred_proba)
roc_auc = auc(fpr, tpr)

ax2.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
ax2.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
ax2.set_xlim([0.0, 1.0])
ax2.set_ylim([0.0, 1.05])
ax2.set_xlabel('False Positive Rate', fontweight='bold')
ax2.set_ylabel('True Positive Rate', fontweight='bold')
ax2.set_title(f'Curva ROC - {best_model_name}', fontsize=12, fontweight='bold')
ax2.legend(loc="lower right")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n✓ Evaluación en Test Set completada")
print(f"\n⭐ AUC Final en Test Set: {test_auc:.4f}")

In [0]:
# Configurar experimento MLflow
experiment_name = f"/Users/{spark.sql('SELECT current_user()').collect()[0][0]}/credit_risk_scoring"
mlflow.set_experiment(experiment_name)

print(f"=== EXPERIMENTO MLFLOW ===")
print(f"Nombre: {experiment_name}")
print(f"\nEntrenando múltiples modelos de clasificación...\n")

# Diccionario para almacenar resultados
results = []

# ========================================
# MODELO 1: LOGISTIC REGRESSION
# ========================================
print("1. Entrenando Logistic Regression...")
with mlflow.start_run(run_name="Logistic_Regression") as run:
    model_lr = LogisticRegression(max_iter=1000, random_state=42)
    model_lr.fit(X_train_processed, y_train)
    
    # Predicciones
    y_train_pred_lr = model_lr.predict_proba(X_train_processed)[:, 1]
    y_val_pred_lr = model_lr.predict_proba(X_val_processed)[:, 1]
    
    # Métricas
    train_auc_lr = roc_auc_score(y_train, y_train_pred_lr)
    val_auc_lr = roc_auc_score(y_val, y_val_pred_lr)
    
    # Log en MLflow
    mlflow.log_param("model_type", "Logistic Regression")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_metric("train_auc", train_auc_lr)
    mlflow.log_metric("val_auc", val_auc_lr)
    mlflow.sklearn.log_model(model_lr, "model")
    
    results.append({
        'modelo': 'Logistic Regression',
        'train_auc': train_auc_lr,
        'val_auc': val_auc_lr,
        'run_id': run.info.run_id
    })
    
    print(f"   Train AUC: {train_auc_lr:.4f}")
    print(f"   Val AUC: {val_auc_lr:.4f}\n")

# ========================================
# MODELO 2: RANDOM FOREST
# ========================================
print("2. Entrenando Random Forest...")
with mlflow.start_run(run_name="Random_Forest") as run:
    model_rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    model_rf.fit(X_train_processed, y_train)
    
    # Predicciones
    y_train_pred_rf = model_rf.predict_proba(X_train_processed)[:, 1]
    y_val_pred_rf = model_rf.predict_proba(X_val_processed)[:, 1]
    
    # Métricas
    train_auc_rf = roc_auc_score(y_train, y_train_pred_rf)
    val_auc_rf = roc_auc_score(y_val, y_val_pred_rf)
    
    # Log en MLflow
    mlflow.log_param("model_type", "Random Forest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 10)
    mlflow.log_metric("train_auc", train_auc_rf)
    mlflow.log_metric("val_auc", val_auc_rf)
    mlflow.sklearn.log_model(model_rf, "model")
    
    results.append({
        'modelo': 'Random Forest',
        'train_auc': train_auc_rf,
        'val_auc': val_auc_rf,
        'run_id': run.info.run_id
    })
    
    print(f"   Train AUC: {train_auc_rf:.4f}")
    print(f"   Val AUC: {val_auc_rf:.4f}\n")

# ========================================
# MODELO 3: GRADIENT BOOSTING
# ========================================
print("3. Entrenando Gradient Boosting...")
with mlflow.start_run(run_name="Gradient_Boosting") as run:
    model_gb = GradientBoostingClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
    model_gb.fit(X_train_processed, y_train)
    
    # Predicciones
    y_train_pred_gb = model_gb.predict_proba(X_train_processed)[:, 1]
    y_val_pred_gb = model_gb.predict_proba(X_val_processed)[:, 1]
    
    # Métricas
    train_auc_gb = roc_auc_score(y_train, y_train_pred_gb)
    val_auc_gb = roc_auc_score(y_val, y_val_pred_gb)
    
    # Log en MLflow
    mlflow.log_param("model_type", "Gradient Boosting")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 5)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_metric("train_auc", train_auc_gb)
    mlflow.log_metric("val_auc", val_auc_gb)
    mlflow.sklearn.log_model(model_gb, "model")
    
    results.append({
        'modelo': 'Gradient Boosting',
        'train_auc': train_auc_gb,
        'val_auc': val_auc_gb,
        'run_id': run.info.run_id
    })
    
    print(f"   Train AUC: {train_auc_gb:.4f}")
    print(f"   Val AUC: {val_auc_gb:.4f}\n")

print("✓ Modelos básicos entrenados")

In [0]:
# ========================================
# MODELO 4: XGBOOST
# ========================================
print("4. Entrenando XGBoost...")
with mlflow.start_run(run_name="XGBoost") as run:
    model_xgb = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='logloss'
    )
    model_xgb.fit(X_train_processed, y_train)
    
    # Predicciones
    y_train_pred_xgb = model_xgb.predict_proba(X_train_processed)[:, 1]
    y_val_pred_xgb = model_xgb.predict_proba(X_val_processed)[:, 1]
    
    # Métricas
    train_auc_xgb = roc_auc_score(y_train, y_train_pred_xgb)
    val_auc_xgb = roc_auc_score(y_val, y_val_pred_xgb)
    
    # Log en MLflow
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("subsample", 0.8)
    mlflow.log_param("colsample_bytree", 0.8)
    mlflow.log_metric("train_auc", train_auc_xgb)
    mlflow.log_metric("val_auc", val_auc_xgb)
    mlflow.xgboost.log_model(model_xgb, "model")
    
    results.append({
        'modelo': 'XGBoost',
        'train_auc': train_auc_xgb,
        'val_auc': val_auc_xgb,
        'run_id': run.info.run_id
    })
    
    print(f"   Train AUC: {train_auc_xgb:.4f}")
    print(f"   Val AUC: {val_auc_xgb:.4f}\n")

# ========================================
# MODELO 5: LIGHTGBM
# ========================================
print("5. Entrenando LightGBM...")
with mlflow.start_run(run_name="LightGBM") as run:
    model_lgb = lgb.LGBMClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbose=-1
    )
    model_lgb.fit(X_train_processed, y_train)
    
    # Predicciones
    y_train_pred_lgb = model_lgb.predict_proba(X_train_processed)[:, 1]
    y_val_pred_lgb = model_lgb.predict_proba(X_val_processed)[:, 1]
    
    # Métricas
    train_auc_lgb = roc_auc_score(y_train, y_train_pred_lgb)
    val_auc_lgb = roc_auc_score(y_val, y_val_pred_lgb)
    
    # Log en MLflow
    mlflow.log_param("model_type", "LightGBM")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("subsample", 0.8)
    mlflow.log_param("colsample_bytree", 0.8)
    mlflow.log_metric("train_auc", train_auc_lgb)
    mlflow.log_metric("val_auc", val_auc_lgb)
    mlflow.lightgbm.log_model(model_lgb, "model")
    
    results.append({
        'modelo': 'LightGBM',
        'train_auc': train_auc_lgb,
        'val_auc': val_auc_lgb,
        'run_id': run.info.run_id
    })
    
    print(f"   Train AUC: {train_auc_lgb:.4f}")
    print(f"   Val AUC: {val_auc_lgb:.4f}\n")

print("✓ Todos los modelos entrenados exitosamente")

In [0]:
# Cargar datos de Silver
df_gold = spark.table("silver.credit_risk_cleaned").toPandas()

# Seleccionar features para modelado (excluir target y flags)
feature_cols = [col for col in df_gold.columns if col not in ['target', 'data_quality_flag']]

# Identificar columnas numéricas y categóricas
numeric_features = df_gold[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
categorical_features = df_gold[feature_cols].select_dtypes(include=['object']).columns.tolist()

print("=== GOLD LAYER - FEATURES FINALES ===")
print(f"\nTotal features: {len(feature_cols)}")
print(f"Features numéricos: {len(numeric_features)}")
print(f"Features categóricos: {len(categorical_features)}")
print(f"\nFeatures numéricos: {numeric_features}")
print(f"\nFeatures categóricos: {categorical_features}")

# Preparar dataset final
df_gold_final = df_gold[feature_cols + ['target']].copy()

# Guardar Gold Layer
df_gold_spark = spark.createDataFrame(df_gold_final)
df_gold_spark.write.format("delta").mode("overwrite").saveAsTable("gold.credit_risk_features")

print(f"\n✓ Gold Layer creada: {df_gold_final.shape}")
print(f"Tasa de default: {df_gold_final['target'].mean():.2%}")

display(df_gold_final.head())

In [0]:
# Cargar datos de Gold Layer
df_model = spark.table("gold.credit_risk_features").toPandas()

# Separar features y target
X = df_model.drop('target', axis=1)
y = df_model['target']

print("=== PREPROCESAMIENTO ===")
print(f"\nShape X: {X.shape}")
print(f"Shape y: {y.shape}")
print(f"\nDistribución target: {y.value_counts().to_dict()}")

# Split train/validation/test (60/20/20)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

print(f"\nTrain set: {X_train.shape[0]} ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validation set: {X_val.shape[0]} ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test set: {X_test.shape[0]} ({X_test.shape[0]/len(X)*100:.1f}%)")

print(f"\nJustificación del split:")
print("- Train (60%): Suficiente data para entrenar modelos complejos")
print("- Validation (20%): Para ajuste de hiperparámetros y selección de modelos")
print("- Test (20%): Hold-out final para evaluación no sesgada")

# Identificar columnas numéricas y categóricas
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print(f"\n✓ Features numéricos: {len(numeric_features)}")
print(f"✓ Features categóricos: {len(categorical_features)}")

# Crear pipelines de preprocesamiento
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from category_encoders import OneHotEncoder

# Pipeline para features numéricos
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# Pipeline para features categóricos
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(use_cat_names=True))
])

# Combinar pipelines
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Aplicar transformaciones
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print(f"\n✓ Datos preprocesados")
print(f"Shape train procesado: {X_train_processed.shape}")
print(f"Shape val procesado: {X_val_processed.shape}")
print(f"Shape test procesado: {X_test_processed.shape}")

In [0]:
# Cargar datos de Silver para EDA
df_eda = spark.table("silver.credit_risk_cleaned").toPandas()

print("=== EXPLORATORY DATA ANALYSIS ===")
print(f"\nShape: {df_eda.shape}")
print(f"\nInformación básica:")
print(df_eda.info())

# Identificar tipos de features
print("\n=== TIPOS DE FEATURES ===")
feature_types = []
for col in df_eda.columns:
    if col in ['target', 'data_quality_flag']:
        continue
    if df_eda[col].dtype in ['int64', 'float64']:
        if df_eda[col].nunique() < 10:
            feature_types.append([col, 'Categórico (numérico)'])
        else:
            feature_types.append([col, 'Numérico continuo'])
    else:
        feature_types.append([col, 'Categórico (string)'])

feature_type_df = pd.DataFrame(feature_types, columns=['Feature', 'Tipo'])
print(feature_type_df.to_string(index=False))

# Estadísticas descriptivas
print("\n=== ESTADÍSTICAS DESCRIPTIVAS ===")
display(df_eda.describe())

# Distribución del target
if 'target' in df_eda.columns:
    plt.figure(figsize=(8, 5))
    target_counts = df_eda['target'].value_counts()
    plt.bar(['No Default (0)', 'Default (1)'], target_counts.values, color=['green', 'red'], alpha=0.7)
    plt.title('Distribución de la Variable Target', fontsize=14, fontweight='bold')
    plt.ylabel('Frecuencia')
    plt.xlabel('Clase')
    for i, v in enumerate(target_counts.values):
        plt.text(i, v + 100, str(v), ha='center', fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f"\nBalance de clases:")
    print(f"No Default (0): {target_counts[0]} ({target_counts[0]/len(df_eda)*100:.1f}%)")
    print(f"Default (1): {target_counts[1]} ({target_counts[1]/len(df_eda)*100:.1f}%)")

# Correlograma de features numéricas
numeric_cols = df_eda.select_dtypes(include=[np.number]).columns.tolist()
if 'target' in numeric_cols:
    numeric_cols.remove('target')
if 'data_quality_flag' in numeric_cols:
    numeric_cols.remove('data_quality_flag')

if len(numeric_cols) > 0:
    plt.figure(figsize=(14, 10))
    correlation_matrix = df_eda[numeric_cols + ['target']].corr()
    sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
                square=True, linewidths=1, cbar_kws={"shrink": 0.8})
    plt.title('Matriz de Correlación - Features vs Target', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Top correlaciones con target
    print("\n=== TOP CORRELACIONES CON TARGET ===")
    target_corr = correlation_matrix['target'].sort_values(ascending=False)
    print(target_corr[target_corr.index != 'target'])

# Visualizaciones: Features vs Target (top 4 más correlacionados)
target_corr_abs = correlation_matrix['target'].abs().sort_values(ascending=False)
top_features = target_corr_abs[target_corr_abs.index != 'target'].head(4).index.tolist()

if len(top_features) >= 2:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.ravel()
    
    for idx, feature in enumerate(top_features[:4]):
        if 'target' in df_eda.columns:
            df_eda.boxplot(column=feature, by='target', ax=axes[idx])
            axes[idx].set_title(f'{feature} vs Target')
            axes[idx].set_xlabel('Target (0=No Default, 1=Default)')
    
    plt.suptitle('Top Features vs Target', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

print("\n✓ EDA completado")

In [0]:
%pip install kaggle scikit-learn xgboost lightgbm optuna category-encoders imbalanced-learn --quiet
dbutils.library.restartPython()

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, roc_curve, auc
from category_encoders import OneHotEncoder
import xgboost as xgb
import lightgbm as lgb
import mlflow
import mlflow.sklearn
import optuna
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✓ Bibliotecas importadas exitosamente")

In [0]:
# Descargar dataset de Kaggle
# Nota: Si no tienes configurado Kaggle API, descarga manualmente desde:
# https://www.kaggle.com/datasets/laotse/credit-risk-dataset/data

try:
    # Intentar descargar con Kaggle API
    import os
    os.makedirs('/tmp/credit_data', exist_ok=True)
    !kaggle datasets download -d laotse/credit-risk-dataset -p /tmp/credit_data --unzip
    data_path = '/tmp/credit_data/credit_risk_dataset.csv'
    print("✓ Dataset descargado con Kaggle API")
except:
    print("⚠ Kaggle API no configurada. Por favor descarga manualmente el archivo.")
    print("Visita: https://www.kaggle.com/datasets/laotse/credit-risk-dataset/data")
    # Alternativa: subir archivo manualmente
    data_path = '/dbfs/FileStore/credit_risk_dataset.csv'  # Ajusta esta ruta

# Cargar datos raw
df_bronze = pd.read_csv(data_path)

# Guardar en Delta como Bronze Layer
df_bronze_spark = spark.createDataFrame(df_bronze)
df_bronze_spark.write.format("delta").mode("overwrite").saveAsTable("bronze.credit_risk_raw")

print(f"✓ Bronze Layer creada: {df_bronze.shape[0]} registros, {df_bronze.shape[1]} columnas")
print(f"\nColumnas: {list(df_bronze.columns)}")
display(df_bronze.head())

In [0]:
# Cargar desde Bronze
df_silver = spark.table("bronze.credit_risk_raw").toPandas()

print("=== ANÁLISIS DE DATOS BRONZE ===")
print(f"Shape: {df_silver.shape}")
print(f"\nValores nulos por columna:")
print(df_silver.isnull().sum())
print(f"\nTipos de datos:")
print(df_silver.dtypes)
print(f"\nDuplicados: {df_silver.duplicated().sum()}")

# Limpieza de datos
# 1. Eliminar duplicados
df_silver = df_silver.drop_duplicates()

# 2. Renombrar columnas para consistencia
df_silver.columns = df_silver.columns.str.lower().str.replace(' ', '_')

# 3. Identificar variable target (loan_status o cb_person_default_on_file)
if 'loan_status' in df_silver.columns:
    target_col = 'loan_status'
    # Convertir a binario: 1 = default (malo), 0 = no default (bueno)
    df_silver['target'] = (df_silver[target_col] == 1).astype(int)
elif 'cb_person_default_on_file' in df_silver.columns:
    # Usar historial de default como proxy
    df_silver['target'] = (df_silver['cb_person_default_on_file'] == 'Y').astype(int)
else:
    # Buscar otra columna de target
    print("\n⚠ Verificar columna target en los datos")

# 4. Manejo de valores atípicos extremos (opcional - mantener registro)
df_silver['data_quality_flag'] = 0  # 0 = ok, 1 = revisión

# 5. Crear features derivadas
if 'person_age' in df_silver.columns and 'person_emp_length' in df_silver.columns:
    df_silver['age_emp_ratio'] = df_silver['person_age'] / (df_silver['person_emp_length'] + 1)

if 'loan_amnt' in df_silver.columns and 'person_income' in df_silver.columns:
    df_silver['debt_to_income'] = df_silver['loan_amnt'] / (df_silver['person_income'] + 1)
    df_silver['income_to_loan'] = df_silver['person_income'] / (df_silver['loan_amnt'] + 1)

if 'loan_int_rate' in df_silver.columns and 'loan_amnt' in df_silver.columns:
    df_silver['total_interest'] = df_silver['loan_amnt'] * df_silver['loan_int_rate'] / 100

# Guardar Silver Layer
df_silver_spark = spark.createDataFrame(df_silver)
df_silver_spark.write.format("delta").mode("overwrite").saveAsTable("silver.credit_risk_cleaned")

print(f"\n✓ Silver Layer creada: {df_silver.shape[0]} registros, {df_silver.shape[1]} columnas")
print(f"\nDistribución del target:")
if 'target' in df_silver.columns:
    print(df_silver['target'].value_counts())
    print(f"Tasa de default: {df_silver['target'].mean():.2%}")

display(df_silver.head())